# CS771 HW3: Colab runner

Choose **Runtime → Change runtime type → GPU** (L4 or A100 recommended, T4 works), then run the cells in order.

The code is cloned from GitHub to fast local disk (`/content/Assignment3`), datasets download to local disk, and **logs/checkpoints are written to Google Drive** (`MyDrive/CS771/Assignment3/logs`) so they survive disconnects.
Keep this tab open: Colab Pro stops idle or closed sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup (re-run this cell after every new session / disconnect)

In [ ]:
REPO = "https://github.com/fluorspar20/ddpm-vs-rectified-flow.git"
LOGS = "/content/drive/MyDrive/CS771/Assignment3/logs"   # checkpoints + logs live on Drive

import os
os.makedirs(LOGS, exist_ok=True)
if os.path.isdir("/content/Assignment3/.git"):
    !git -C /content/Assignment3 pull -q
else:
    !git clone -q {REPO} /content/Assignment3
!ln -sfn "{LOGS}" /content/Assignment3/logs
assert os.path.isdir("/content/Assignment3/code"), "clone failed - check the output above"
%cd /content/Assignment3/code
!pip install -q -r requirements.txt
!ls -la ../logs ../pretrained

## 2. MNIST (4 runs, 30 epochs each)
Run one cell at a time. Output folders: `logs/<config>_final/` (checkpoints every 5 epochs, `sample-<epoch>.png`, `model_final.pth.tar`).

In [ ]:
!python train.py configs/mnist_ddpm_unet.yaml --output final -p 500

In [ ]:
!python train.py configs/mnist_ddpm_vit.yaml --output final -p 500

In [ ]:
!python train.py configs/mnist_fm_unet.yaml --output final -p 500

In [ ]:
!python train.py configs/mnist_fm_vit.yaml --output final -p 500

**Optional (A100/L4 only): run two jobs at once in the background.** The MNIST models are small, so a big GPU has spare capacity.
Check progress with `!tail -n 3 ../logs/*.log`.

In [ ]:
# !nohup python train.py configs/mnist_fm_unet.yaml --output final -p 500 > ../logs/mnist_fm_unet_final.log 2>&1 &
# !nohup python train.py configs/mnist_fm_vit.yaml  --output final -p 500 > ../logs/mnist_fm_vit_final.log  2>&1 &

## 3. MiniImageNet latent FM (2 runs, 80 epochs each). The first run downloads the dataset (several GB) to local disk.

In [ ]:
!python train.py configs/miniimagenet_fm_unet.yaml --output final -p 500

In [ ]:
!python train.py configs/miniimagenet_fm_vit.yaml --output final -p 500

### If a session dies mid-training: resume
Re-run the Setup cell, then resume from the latest `epoch_XXX.pth.tar` in that run's folder (same `--output final`).

In [ ]:
# !ls ../logs/miniimagenet_fm_unet_final
# !python train.py configs/miniimagenet_fm_unet.yaml --output final -p 500 --resume ../logs/miniimagenet_fm_unet_final/epoch_049.pth.tar

## 4. FID evaluation (required baseline: **no CFG**)
`eval.py` loads the EMA weights of `model_final.pth.tar`. The config default turns CFG on, so `--no-cfg` gives the required baseline.
Save each printed FID for the writeup (the cell also appends it to `logs/fid_results.txt`).

In [ ]:
runs = ["mnist_ddpm_unet", "mnist_ddpm_vit", "mnist_fm_unet", "mnist_fm_vit",
        "miniimagenet_fm_unet", "miniimagenet_fm_vit"]
for r in runs:
    print("=" * 20, r)
    !python eval.py configs/{r}.yaml ../logs/{r}_final --no-cfg 2>&1 | grep -E "FID|Saving" | sed "s/^/{r} no-cfg: /" | tee -a ../logs/fid_results.txt

## 5. Bonus: classifier-free guidance (no retraining). Sweep the scale and compare to the no-CFG baseline.

In [ ]:
for r in ["miniimagenet_fm_unet", "miniimagenet_fm_vit"]:
    for s in [1.5, 2.5, 4.0]:
        !python eval.py configs/{r}.yaml ../logs/{r}_final --cfg-scale {s} 2>&1 | grep FID | sed "s/^/{r} cfg={s}: /" | tee -a ../logs/fid_results.txt

## 6. Training curves (TensorBoard) and results

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../logs

In [ ]:
!cat ../logs/fid_results.txt

## 7. Free Drive space when everything is done
Each intermediate checkpoint also stores optimizer state. This keeps `model_final.pth.tar`, the last epoch checkpoint and the sample grids, and deletes the rest.

In [ ]:
import glob, os
for d in glob.glob("../logs/*_final"):
    ck = sorted(glob.glob(d + "/epoch_*.pth.tar"))
    for f in ck[:-1]:
        os.remove(f); print("removed", f)